In [12]:
import re
from pathlib import Path

import numpy as np
import pandas as pd


# find the repo root (the folder that contains data/) no matter where the notebook runs from
ROOT = Path.cwd().resolve()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

RAW_PATH = ROOT / "data/raw/movies_metadata.csv"
OUT_PATH = ROOT / "data/processed/movie_text.csv"

TEXT_COLS = ["title", "original_title", "overview", "original_language", "status"]

pd.set_option("display.max_colwidth", 80)  
pd.set_option("display.width", 200)

stats = {}   # we'll collect key numbers here for the written report 

In [13]:
raw = pd.read_csv(RAW_PATH, dtype=str, keep_default_na=False)
stats["rows_raw"] = len(raw)
print("rows, columns:", raw.shape)
raw.head(3)

rows, columns: (45466, 24)


,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', 'poster_path': '/7G9915LfUQ2lV...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, 'name': 'Comedy'}, {'id': 10751...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his room until Andy's birthday bri...",...,1995-10-30,373554033,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,,Toy Story,False,7.7,5415
1,False,,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, 'name': 'Fantasy'}, {'id': 1075...",,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an enchanted board game that opens the...,...,1995-12-15,262797249,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso_639_1': 'fr', 'name': 'França...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collection', 'poster_path': '/nLvUdqg...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, 'name': 'Comedy'}]",,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud between next-door neighbors and ...,...,1995-12-22,0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for Love.,Grumpier Old Men,False,6.5,92


In [14]:
id_num = pd.to_numeric(raw["id"],errors="coerce")
bad_id = raw[id_num.isna()]
stats["rows_bad_id"] = len(bad_id)
print("rows with a non-numeric id:", len(bad_id))
bad_id[["adult","id","original_title","overview"]]

rows with a non-numeric id: 3


,adult,id,original_title,overview
19730,- Written by Ørnås,1997-08-20,"[{'iso_639_1': 'en', 'name': 'English'}]",Released
29503,Rune Balot goes to a casino connected to the October corporation to try to ...,2012-09-29,"[{'iso_639_1': 'ja', 'name': '日本語'}]",Released
35587,Avalanche Sharks tells the story of a bikini contest that turns into a horr...,2014-01-01,"[{'iso_639_1': 'en', 'name': 'English'}]",Released


Converts all Ids to a number, if it can't be read as a number we print them out to decide what to do with them.

In [15]:
df = raw[id_num.notna()].copy()
df["id"] = id_num[id_num.notna()].astype("int64")
print("rows after dropping bad ids:", len(df))


rows after dropping bad ids: 45463


We decided to drop these rows, since they do not have a valid ID anywhere, all three have dates instead, so we can safely justify dropping them.

In [16]:
dup_mask = df["id"].duplicated(keep=False)          # True for EVERY row that shares its id with another
print("rows involved in duplicate ids:", dup_mask.sum())
print("extra rows (would be removed):", df["id"].duplicated().sum())

# Are the duplicated rows identical in the text columns, or do they differ?
variants = df[dup_mask].groupby("id")[TEXT_COLS].nunique(dropna=False).max(axis=1)
print("duplicated ids whose text differs between copies:", (variants > 1).sum())

df[dup_mask].sort_values("id")[["id", "title", "original_title", "release_date"]].head(10)

rows involved in duplicate ids: 59
extra rows (would be removed): 30
duplicated ids whose text differs between copies: 0


,id,title,original_title,release_date
5865,4912,Confessions of a Dangerous Mind,Confessions of a Dangerous Mind,2002-12-30
33826,4912,Confessions of a Dangerous Mind,Confessions of a Dangerous Mind,2002-12-30
9165,5511,Le Samouraï,Le Samouraï,1967-10-25
7345,5511,Le Samouraï,Le Samouraï,1967-10-25
44821,10991,Pokémon: Spell of the Unknown,Pokémon 3: The Movie,2000-07-08
4114,10991,Pokémon: Spell of the Unknown,Pokémon 3: The Movie,2000-07-08
14012,11115,Deal,Deal,2008-01-29
24844,11115,Deal,Deal,2008-01-29
44826,12600,Pokémon 4Ever: Celebi - Voice of the Forest,劇場版ポケットモンスター セレビィ 時を越えた遭遇（であい）,2001-07-06
5535,12600,Pokémon 4Ever: Celebi - Voice of the Forest,劇場版ポケットモンスター セレビィ 時を越えた遭遇（であい）,2001-07-06


In [17]:
df["_filled"] = (df[TEXT_COLS] != "").sum(axis=1)     # how many of the 6 columns are non-empty
df = (df.sort_values(["id", "_filled"], ascending=[True, False], kind="stable")
        .drop_duplicates(subset="id", keep="first")
        .drop(columns="_filled")
        .reset_index(drop=True))

stats["rows_dup_id_removed"] = stats["rows_raw"] - stats["rows_bad_id"] - len(df)
print("rows now:", len(df), "| id unique?", df["id"].is_unique)

# Working table: id + the 6 text columns. release_date is kept TEMPORARILY, only to help us
# investigate duplicate titles later. It is dropped before saving.
work = df[["id"] + TEXT_COLS + ["release_date"]].copy()

rows now: 45433 | id unique? True


For each id it keeps the copy with the most non-empty text columns. Ties go to the first one, because the sort is stable. That leaves 45,433 rows with unique ids. The cell also builds work, the id plus the six text columns plus release_date. release_date is only kept temporarily, to investigate duplicate titles.

In [18]:
MESSY_WS = r"^\s|\s$|\s{2,}|[\u00a0\u200b\u200c\u200d\ufeff\t\n\r]"

def audit(frame):
    rows = {}
    for c in TEXT_COLS:
        s = frame[c]
        rows[c] = {
            "missing (NaN)":    int(s.isna().sum()),
            "empty/blank":      int((s.notna() & (s.str.strip() == "")).sum()),
            "messy whitespace": int(s.str.contains(MESSY_WS, regex=True, na=False).sum()),
            "n_unique":         int(s.nunique()),
        }
    return pd.DataFrame(rows).T

audit_before = audit(work)
audit_before

,missing (NaN),empty/blank,messy whitespace,n_unique
title,0,3,22,42278
original_title,0,0,37,43371
overview,0,959,1439,44307
original_language,0,11,0,90
status,0,84,0,7


The whitespace checker, the MESSY_WS computes four numbers for each column (this was claude generated fully) but it flags anything messy white space related.

In [19]:
for c in ["title", "original_title", "overview"]:
    print(f"--- {c}: most common values (lower-cased, trimmed) ---")
    print(work[c].str.strip().str.lower().value_counts().head(30))
    print()

--- title: most common values (lower-cased, trimmed) ---
title
cinderella                       11
hamlet                            9
alice in wonderland               9
beauty and the beast              8
les misérables                    8
the three musketeers              7
treasure island                   7
a christmas carol                 7
wuthering heights                 6
bluebeard                         6
the hound of the baskervilles     6
the journey                       6
macbeth                           6
the circle                        6
the hunters                       6
the stranger                      6
jane eyre                         6
mother                            6
countdown                         6
heidi                             6
aftermath                         6
first love                        6
love                              6
home                              6
the forest                        6
blind date                        5
t

KeyError: 'tagline'

By printing the most common values we check for any unique possible title placeholders, none are found so we move on.

In [ ]:
INVISIBLE = re.compile(r"[\u200b\u200c\u200d\ufeff]")

def clean_whitespace(value):
    if pd.isna(value):                 # leave missing values alone
        return value
    value = INVISIBLE.sub("", value)
    value = re.sub(r"\s+", " ", value) # any run of whitespace -> one space
    return value.strip()

before = work[TEXT_COLS].copy()
for c in TEXT_COLS:
    work[c] = work[c].map(clean_whitespace)

changed = (before.ne(work[TEXT_COLS]) & before.notna()).sum()
stats["whitespace_changed"] = changed.to_dict()
print("values changed by whitespace cleaning:")
changed

values changed by whitespace cleaning:


title                  22
original_title         79
overview             1439
tagline               113
original_language       0
status                  0
dtype: int64

In [ ]:
GENERIC = {"", "n/a", "na", "nan", "null", "none", "-", "--", "?", "tbd"}
DESCRIPTION_ONLY = {"no overview found", "no overview", "no overview available",
                    "no description", "unknown", "not available"}

PLACEHOLDERS = {
    "title":             GENERIC,
    "original_title":    GENERIC,
    "overview":          GENERIC | DESCRIPTION_ONLY,
    "original_language": GENERIC,
    "status":            GENERIC,
}

def normalize_key(value):
    return value.strip().lower().rstrip(".!")

def placeholder_mask(series, allowed):
    keys = series.map(lambda v: normalize_key(v) if isinstance(v, str) else None)
    return keys.isin(allowed)

for c in TEXT_COLS:
    m = placeholder_mask(work[c], PLACEHOLDERS[c])
    print(f"--- {c}: {m.sum()} values match a placeholder ---")
    if m.sum():
        print(work.loc[m, c].value_counts().head(8).to_string())
    print()

--- title: 3 values match a placeholder ---
title
    3

--- original_title: 0 values match a placeholder ---

--- overview: 1104 values match a placeholder ---
overview
                      959
No overview found.    133
No Overview             7
No overview             1
No overview found       1
...                     1
No overview.            1
Not Available           1

--- tagline: 25037 values match a placeholder ---
tagline
     25033
-        4

--- original_language: 11 values match a placeholder ---
original_language
    11

--- status: 84 values match a placeholder ---
status
    84



Now we actually clean based on possible placeholders, most of them would share similar names so we use a set. For the description specific ones we have a seperate list. With this we get a mask that will be used to replace with NaN later. The biggest reason we seperate is because of the possibility of a film being named something like "Unknown"

In [ ]:
replaced = {}
for c in TEXT_COLS:
    m = placeholder_mask(work[c], PLACEHOLDERS[c])
    replaced[c] = int(m.sum())
    work[c] = work[c].mask(m)          # .mask(True-rows) -> sets those values to NaN

stats["placeholders_to_nan"] = replaced
pd.Series(replaced, name="values set to NaN")

title                    3
original_title           0
overview              1104
tagline              25037
original_language       11
status                  84
Name: values set to NaN, dtype: int64

In [ ]:
work["original_language"] = work["original_language"].str.lower()

valid = work["original_language"].str.fullmatch(r"[a-z]{2}", na=False)
odd_lang = work[work["original_language"].notna() & ~valid]
print("non-2-letter language values:", len(odd_lang))
print(odd_lang["original_language"].value_counts().head())


non-2-letter language values: 0
Series([], Name: count, dtype: int64)


Normalizes the languages by ensuring their all lowercase. No non 2 letter values means we can move on.

In [ ]:
CANONICAL_STATUS = ["Released", "Post Production", "In Production", "Planned", "Rumored", "Canceled"]
status_map = {s.lower(): s for s in CANONICAL_STATUS}

mapped = work["status"].str.lower().map(status_map)
unexpected = work.loc[work["status"].notna() & mapped.isna(), "status"].unique()
print("unexpected status values:", unexpected)


unexpected status values: <StringArray>
[]
Length: 0, dtype: str


Pulled TMDB status values, since there is no anamolys we move on.

In [ ]:
both = work["title"].notna() & work["original_title"].notna()
same = both & (work["title"] == work["original_title"])
diff = both & ~same
case_only = diff & (work["title"].str.casefold() == work["original_title"].str.casefold())

comparison = {
    "both present":                      int(both.sum()),
    "identical":                         int(same.sum()),
    "different":                         int(diff.sum()),
    "  of which differ only by case":    int(case_only.sum()),
    "title missing, original present":   int((work["title"].isna() & work["original_title"].notna()).sum()),
    "original missing, title present":   int((work["title"].notna() & work["original_title"].isna()).sum()),
    "both missing":                      int((work["title"].isna() & work["original_title"].isna()).sum()),
}
stats["title_vs_original"] = comparison
pd.Series(comparison, name="rows")

both present                       45430
identical                          34042
different                          11388
  of which differ only by case        82
title missing, original present        3
original missing, title present        0
both missing                           0
Name: rows, dtype: int64

In [ ]:
print("Share identical: {:.1%}".format(same.sum() / both.sum()))

lang = work["original_language"].fillna("missing")
print("\nWhere do the differences come from? (top languages among differing rows)")
print(lang[diff].value_counts().head(8))

print("\nShare of English-original films where title != original_title: {:.2%}".format(
      diff[lang == "en"].mean()))
print("Share of non-English-original films where title != original_title: {:.2%}".format(
      diff[lang != "en"].mean()))

Share identical: 74.9%

Where do the differences come from? (top languages among differing rows)
original_language
en    1708
fr    1687
ja    1267
it    1061
ru     775
es     652
de     645
ko     431
Name: count, dtype: int64

Share of English-original films where title != original_title: 5.30%
Share of non-English-original films where title != original_title: 73.43%


In [ ]:
cols = ["id", "title", "original_title", "original_language"]
print("Differ, original language NOT English (expected: translated titles):")
display(work[diff & (lang != "en")][cols].sample(frac=1, random_state=42).head(8))

print("Differ, original language IS English (worth a look: alternate titles / typos?):")
display(work[diff & (lang == "en")][cols].sample(frac=1, random_state=42).head(8))

Differ, original language NOT English (expected: translated titles):


,id,title,original_title,original_language
20553,51281,The Ugly Duckling,Гадкий утёнок,ru
5283,11945,Emil in Lonneberga,Emil i Lönneberga,sv
33994,156045,The Imp,凶榜,zh
38718,255746,The Reunion 2 - The Funeral,Klassefesten 2 - begravelsen,da
5261,11915,The Gendarme and the Gendarmettes,Le gendarme et les gendarmettes,fr
23934,65545,Vanished Empire,Исчезнувшая империя,ru
22965,61123,Vares - The Girls of April,Vares – Huhtikuun tytöt,fi
18073,43766,Fireworks Wednesday,چهارشنبه سوری,fa


Differ, original language IS English (worth a look: alternate titles / typos?):


,id,title,original_title,original_language
20164,49830,Bio Slime,Contagion,en
35386,180651,Fists of Legend,전설의 주먹,en
35816,190496,Insatiability,Nienasycenie,en
7242,15725,Chasing Ghosts: Beyond the Arcade,Chasing Ghosts,en
18296,44142,Ginger & Cinnamon,Dillo con parole mie,en
36167,197790,Ninja USA,猛龍煞星,en
18869,45756,Saint,Sint,en
21227,53776,Nobody Else But You,Poupoupidou,en


In [ ]:
has_title = work["title"].notna()
dup_title = has_title & work["title"].duplicated(keep=False)

stats["dup_title_rows"]  = int(dup_title.sum())
stats["dup_title_names"] = int(work.loc[dup_title, "title"].nunique())
print("rows whose title is shared with another row:", stats["dup_title_rows"])
print("distinct repeated titles:", stats["dup_title_names"])
print()
top_titles = work.loc[dup_title, "title"].value_counts().head(10)
top_titles

rows whose title is shared with another row: 5460
distinct repeated titles: 2307



title
Cinderella              11
Hamlet                   9
Alice in Wonderland      9
Beauty and the Beast     8
Les Misérables           8
The Three Musketeers     7
Treasure Island          7
A Christmas Carol        7
Wuthering Heights        6
Bluebeard                6
Name: count, dtype: int64

Since there are so many titles with different original titles to current, it proves they aren't redundant and we should keep both rows as is.

In [ ]:
example = top_titles.index[0]
print(f"All rows titled {example!r}: different ids -> different movies")
work[work["title"] == example][["id", "title", "original_title", "original_language", "release_date"]].head(10)

All rows titled 'Cinderella': different ids -> different movies


,id,title,original_title,original_language,release_date
4749,11224,Cinderella,Cinderella,en,1950-03-04
17475,42651,Cinderella,Zolushka,ru,1947-11-28
17604,42884,Cinderella,Cinderella,en,1997-11-02
18412,44459,Cinderella,Cinderella,en,1957-03-31
28538,92349,Cinderella,Cinderella,en,1914-12-28
29252,98604,Cinderella,Zolushka,ru,2012-02-14
30011,105875,Cinderella,Cinderella,en,2002-08-06
30828,114108,Cinderella,Cendrillon,fr,1899-10-01
33741,150689,Cinderella,Cinderella,en,2015-03-12
39118,261985,Cinderella,Cenerentola,en,2011-10-30


In [ ]:
same_overview = work[work["title"].notna() & work["overview"].notna()]
same_overview = same_overview[same_overview.duplicated(subset=["title", "overview"], keep=False)]

same_date = work[work["title"].notna() & work["release_date"].ne("")]
same_date = same_date[same_date.duplicated(subset=["title", "release_date"], keep=False)]

stats["dup_title_same_overview_rows"] = len(same_overview)
stats["dup_title_same_date_rows"]     = len(same_date)
print("same title + same overview (rows):", len(same_overview))
print("same title + same release date (rows):", len(same_date))

same_overview.sort_values("title")[["id", "title", "release_date", "overview"]].head(10)

same title + same overview (rows): 20
same title + same release date (rows): 0


,id,title,release_date,overview
13429,31648,12 Chairs,1971-06-21,A former aristocrat Ippolit Vorobyaninov leads a miserable life in Soviet Ru...
12027,27934,12 Chairs,1976-12-31,A former aristocrat Ippolit Vorobyaninov leads a miserable life in Soviet Ru...
12384,28769,A Christmas Carol,2004-11-28,Miser Ebenezer Scrooge is awakened on Christmas Eve by spirits who reveal to...
8163,17979,A Christmas Carol,2009-11-04,Miser Ebenezer Scrooge is awakened on Christmas Eve by spirits who reveal to...
32557,133021,A Streetcar Named Desire,1995-10-24,Blanche Dubois goes to visit her pregnant sister and husband Stanley in New ...
32556,133018,A Streetcar Named Desire,1984-03-04,Blanche Dubois goes to visit her pregnant sister and husband Stanley in New ...
3771,9986,Charlotte's Web,2006-12-15,"Wilbur the pig is scared of the end of the season, because he knows that com..."
6988,15171,Charlotte's Web,1973-03-01,"Wilbur the pig is scared of the end of the season, because he knows that com..."
17372,42495,King Lear,1971-02-04,"King Lear, old and tired, divides his kingdom among his daughters, giving gr..."
31360,119840,King Lear,1982-09-19,"King Lear, old and tired, divides his kingdom among his daughters, giving gr..."


In [ ]:
assert work["id"].is_unique, "ids are not unique"
assert work["id"].notna().all(), "missing ids"

for c in TEXT_COLS:
    s = work[c].dropna()
    assert not s.str.contains(MESSY_WS, regex=True).any(), f"{c}: messy whitespace remains"
    assert not (s.str.strip() == "").any(), f"{c}: blank strings remain"
    assert not placeholder_mask(s, PLACEHOLDERS[c]).any(), f"{c}: placeholders remain"

print("All checks passed.")

All checks passed.


Duplicate titles should be treated as different rows, and NOT merged, unless they share IDs. This is because there can be different versions, re-releases, or remakes of a movie. As such we keep them.

In [ ]:
audit_after = audit(work)
print("BEFORE cleaning"); display(audit_before)
print("AFTER cleaning");  display(audit_after)

NameError: name 'audit' is not defined

In [ ]:
final = work[["id"] + TEXT_COLS].copy()
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
final.to_csv(OUT_PATH, index=False)

stats["rows_final"] = len(final)
stats["missing_final"] = final[TEXT_COLS].isna().sum().to_dict()
print("saved:", OUT_PATH.resolve())
print("shape:", final.shape)
final.head()

NameError: name 'work' is not defined

In [92]:
import json
print(json.dumps(stats, indent=2))

{
  "rows_raw": 45466,
  "rows_bad_id": 3,
  "rows_dup_id_removed": 30,
  "whitespace_changed": {
    "title": 22,
    "original_title": 79,
    "overview": 1439,
    "tagline": 113,
    "original_language": 0,
    "status": 0
  },
  "placeholders_to_nan": {
    "title": 3,
    "original_title": 0,
    "overview": 1104,
    "tagline": 25037,
    "original_language": 11,
    "status": 84
  },
  "title_vs_original": {
    "both present": 45430,
    "identical": 34042,
    "different": 11388,
    "  of which differ only by case": 82,
    "title missing, original present": 3,
    "original missing, title present": 0,
    "both missing": 0
  },
  "dup_title_rows": 5460,
  "dup_title_names": 2307,
  "dup_title_same_overview_rows": 20,
  "dup_title_same_date_rows": 0,
  "rows_final": 45433,
  "missing_final": {
    "title": 3,
    "original_title": 0,
    "overview": 1104,
    "tagline": 25037,
    "original_language": 11,
    "status": 84
  }
}
